In [1]:
import numpy as np
import pandas as pd

### Importing Raw Datasets

In [2]:
raw_crime = pd.read_csv('datasets/raw_crime_2024.csv', sep=';')
raw_nbh = pd.read_excel('datasets/raw_nbh_2024.xlsx')

In [3]:
raw_nbh.shape

(18310, 126)

### Check how many 'BU' for Buurten

In [4]:
print("Crime Cols:", raw_crime.columns)
print("NBH Cols:", raw_nbh.columns)

Crime Cols: Index(['ID', 'SoortMisdrijf', 'WijkenEnBuurten', 'Perioden',
       'GeregistreerdeMisdrijven_1', 'Gemeentenaam_2', 'SoortRegio_3'],
      dtype='object')
NBH Cols: Index(['gwb_code_10', 'gwb_code_8', 'regio', 'gm_naam', 'recs', 'gwb_code',
       'ind_wbi', 'a_inw', 'a_man', 'a_vrouw',
       ...
       'g_afs_kv', 'g_afs_sc', 'g_3km_sc', 'a_opp_ha', 'a_lan_ha', 'a_wat_ha',
       'pst_mvp', 'pst_dekp', 'ste_mvs', 'ste_oad'],
      dtype='object', length=126)


In [5]:
# Crime
# Count rows by region code prefix (GM = municipality, WK = district, BU = neighborhood)
col = 'WijkenEnBuurten'
s = raw_crime[col].astype(str).str.strip()
print('GM_crime:', s.str.startswith('GM', na=False).sum())
print('WK_crime:', s.str.startswith('WK', na=False).sum())
print('BU_crime:', s.str.startswith('BU', na=False).sum())
print("Total Crime Rows", raw_crime.shape[0], "\n")

# NBH
# Count rows by region code prefix (GM = municipality, WK = district, BU = neighborhood)
col = 'gwb_code_10'
s = raw_nbh[col].astype(str).str.strip()
print('GM_nbh:', s.str.startswith('GM', na=False).sum())
print('WK_nbh:', s.str.startswith('WK', na=False).sum())
print('BU_nbh:', s.str.startswith('BU', na=False).sum())
print("Total NBH Rows", raw_nbh.shape[0])

GM_crime: 343
WK_crime: 3424
BU_crime: 14730
Total Crime Rows 18498 

GM_nbh: 342
WK_nbh: 3393
BU_nbh: 14574
Total NBH Rows 18310


### Remove 'GM' and 'WK' from 'WijkenEnBuurten' and 'gwb_code_10'

In [6]:
# Keep only neighborhood (BU) level: drop rows where code starts with 'GM' or 'WK'
crime = raw_crime[~raw_crime['WijkenEnBuurten'].astype(str).str.strip().str.startswith(('GM', 'WK'), na=False)]
nbh = raw_nbh[~raw_nbh['gwb_code_10'].astype(str).str.strip().str.startswith(('GM', 'WK'), na=False)]

# Crime
# Count rows by region code prefix (GM = municipality, WK = district, BU = neighborhood)
col = 'WijkenEnBuurten'
s = crime[col].astype(str).str.strip()
print('GM_crime:', s.str.startswith('GM', na=False).sum())
print('WK_crime:', s.str.startswith('WK', na=False).sum())
print('BU_crime:', s.str.startswith('BU', na=False).sum())
print("Total Crime Rows", crime.shape[0], "\n")

# NBH
# Count rows by region code prefix (GM = municipality, WK = district, BU = neighborhood)
col = 'gwb_code_10'
s = nbh[col].astype(str).str.strip()
print('GM_nbh:', s.str.startswith('GM', na=False).sum())
print('WK_nbh:', s.str.startswith('WK', na=False).sum())
print('BU_nbh:', s.str.startswith('BU', na=False).sum())
print("Total NBH Rows", nbh.shape[0])

GM_crime: 0
WK_crime: 0
BU_crime: 14730
Total Crime Rows 14731 

GM_nbh: 0
WK_nbh: 0
BU_nbh: 14574
Total NBH Rows 14575


### Remove one remaining row which starts with 'NL'

In [7]:
# drop rows where code starts with 'NL'
crime = crime[~crime['WijkenEnBuurten'].astype(str).str.strip().str.startswith(('NL'), na=False)]
nbh = nbh[~nbh['gwb_code_10'].astype(str).str.strip().str.startswith(('NL'), na=False)]

print("Total Crime Rows", crime.shape[0])
print("Total NBH Rows", nbh.shape[0])

Total Crime Rows 14730
Total NBH Rows 14574


### Final Raw Dataset

In [8]:
print("Total Crime Rows", crime.shape[0])
print("Total NBH Rows", nbh.shape[0])

Total Crime Rows 14730
Total NBH Rows 14574


## Merging datasets

### Checking missing values - 0

In [9]:
crime["WijkenEnBuurten"].isna().mean(), nbh["gwb_code_10"].isna().mean()

(0.0, 0.0)

### Standardize & Create join keys

In [10]:
def clean_key(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.strip()
         .str.replace(r"\s+", "", regex=True)  # remove internal whitespace too
         .str.upper()
         .replace({"NAN": np.nan, "NONE": np.nan})
    )

crime = crime.copy()
nbh   = nbh.copy()

crime["merge_key"] = clean_key(crime["WijkenEnBuurten"])
nbh["merge_key"]   = clean_key(nbh["gwb_code_10"])

crime["merge_key"].head(), nbh["merge_key"].head()


(3    BU16800000
 4    BU16800009
 6    BU16800100
 7    BU16800109
 9    BU16800200
 Name: merge_key, dtype: object,
 3    BU00140000
 4    BU00140001
 5    BU00140002
 6    BU00140003
 7    BU00140004
 Name: merge_key, dtype: object)

### Check for Duplicates

In [11]:
crime_key_dupes = crime["merge_key"].duplicated(keep=False).mean()
nbh_key_dupes   = nbh["merge_key"].duplicated(keep=False).mean()

crime_key_dupes, nbh_key_dupes

(0.0, 0.0)

In [12]:
crime["merge_key"].value_counts().head(10)
nbh["merge_key"].value_counts().head(10)


merge_key
BU00140000    1
BU09831307    1
BU09810005    1
BU09810006    1
BU09810007    1
BU09810008    1
BU09810009    1
BU09810100    1
BU09831101    1
BU09831102    1
Name: count, dtype: int64

### Merging

In [13]:
merged = nbh.merge(
    crime.drop(columns=["WijkenEnBuurten"], errors="ignore"),
    on="merge_key",
    how="left",
    validate="one_to_one"
)

merged.shape


(14574, 133)

### Check merging percentage - 99.45%

In [14]:
match_rate = merged["GeregistreerdeMisdrijven_1"].notna().mean()
match_rate


0.9945107726087553

### Remove unmerged rows - 80

In [15]:
unmatched = merged.loc[merged["GeregistreerdeMisdrijven_1"].isna(), "merge_key"]
unmatched.head(), unmatched.nunique()

# Remove the unmerged rows
merged = merged[merged["GeregistreerdeMisdrijven_1"].notna()].reset_index(drop=True)

### Merge neighborhood geometry (GeoPackage)

Join official CBS / PDOK **wijkenbuurten** polygons on the same `merge_key` as above (`buurtcode` standardized ↔ `gwb_code_10`). The CSV export in the next section stays tabular-only; geometry is written to a separate GeoPackage for downstream use (e.g. notebook 4 pre-processing).

In [16]:
import geopandas as gpd

# --- Edit paths if your file lives elsewhere ---
GPKG_PATH = "datasets/wijkenbuurten_2024.gpkg"
GPKG_LAYER = "buurten"  # layer name when the file has multiple layers (e.g. buurten, wijken, gemeenten)
GEO_KEY_COL = "buurtcode"

geo = gpd.read_file(GPKG_PATH, layer=GPKG_LAYER)
geo_small = geo[[GEO_KEY_COL, "geometry"]].copy()
geo_small["merge_key"] = clean_key(geo_small[GEO_KEY_COL])

merged_geom = merged.merge(
    geo_small[["merge_key", "geometry"]],
    on="merge_key",
    how="left",
)
merged_gdf = gpd.GeoDataFrame(merged_geom, geometry="geometry", crs=geo.crs)

n = len(merged_gdf)
matched = merged_gdf.geometry.notna().sum()
print(f"Geometry rows matched: {matched} / {n} ({100 * matched / n:.2f}%)")

Geometry rows matched: 14494 / 14494 (100.00%)


### Export final merged dataset

In [17]:
final = merged.copy()

out_path = "datasets/merged_crime_nbh_2024.csv"
final.to_csv(out_path, index=False)
out_path

'datasets/merged_crime_nbh_2024.csv'

### Export merged dataset with geometry (GeoPackage)

Notebook 4 attaches these polygons via this file instead of merging the CBS source `.gpkg` again.

In [18]:
gpkg_out = "datasets/merged_crime_nbh_2024.gpkg"
merged_gdf.to_file(gpkg_out, driver="GPKG")
gpkg_out

'datasets/merged_crime_nbh_2024.gpkg'